# Eksperimen 7: Uji Generalisasi Lintas Pertanyaan (Leave-One-Question-Out)

**GEMASTIK KTI 2026** | Tim Peneliti

Eksperimen ini dirancang untuk memastikan bahwa model tidak sekadar menghafal jawaban pada data pelatihan (overfitting), melainkan benar-benar mampu mereplikasi logika penilaian manusia secara fundamental. Pengujian dieksekusi dengan mendedikasikan satu pertanyaan secara utuh sebagai data uji yang belum pernah dilihat oleh model, sementara pertanyaan sisanya difungsikan sebagai data latih.

## 0. Persiapan Lingkungan dan Konfigurasi

In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("Lingkungan Eksekusi: Google Colab")
    if not os.path.exists("/content/indo-asag"):
        os.system("git clone https://github.com/Rnov24/indo-asag.git /content/indo-asag")
    os.system("pip install -q -e /content/indo-asag[all]")
    REPO_ROOT = "/content/indo-asag"
except ImportError:
    IN_COLAB = False
    try:
        REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
    except NameError:
        REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    print(f"Lingkungan Eksekusi: Lokal ({REPO_ROOT})")

sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from indo_asag.data import load_dataset
from indo_asag.features import get_feature_names
from indo_asag.evaluation import run_loqo
from indo_asag.utils import load_config
from indo_asag.utils.github import auto_push_to_github

config = load_config(os.path.join(REPO_ROOT, "configs", "base.yaml"))
RESULTS_DIR = os.path.join(REPO_ROOT, "results", "prelim")
PREDS_DIR = os.path.join(RESULTS_DIR, "predictions")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

## 1. Pemuatan Dataset dan Fitur

In [ ]:
DATA_PATH = os.path.join(REPO_ROOT, config["data"]["path"])
df = load_dataset(DATA_PATH)

hc_cols = get_feature_names()
feat_df = pd.read_csv(os.path.join(PREDS_DIR, "features_hc.csv"))
for col in hc_cols:
    df[col] = feat_df[col]

## 2. Eksekusi Eksperimen 7 (LOQO)

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

print("\n" + "=" * 60)
print("EXP 7: Leave-One-Question-Out (LOQO)")
print("=" * 60)

def loqo_predict(train_df, test_df):
    sc = StandardScaler()
    Xt = sc.fit_transform(train_df[hc_cols].values)
    Xv = sc.transform(test_df[hc_cols].values)
    
    svr = SVR(
        kernel=config["model"]["svr"]["kernel"],
        C=config["model"]["svr"]["C"],
        gamma=config["model"]["svr"]["gamma"],
        epsilon=config["model"]["svr"]["epsilon"],
    )
    svr.fit(Xt, train_df["score_norm"].values)
    return svr.predict(Xv)

loqo_df = run_loqo(df, loqo_predict)
print(f"\nRata-rata QWK LOQO: {loqo_df['QWK'].mean():.4f} +/- {loqo_df['QWK'].std():.4f}")

## 3. Penyimpanan Metrik LOQO

In [ ]:
loqo_df.to_csv(os.path.join(METRICS_DIR, "loqo_results.csv"), index=False)
print("[OK] Hasil uji LOQO berhasil disimpan.")

## 4. Publikasi Otomatis ke GitHub

In [ ]:
auto_push_to_github("chore(auto): menyimpan metrik Eksperimen 7 LOQO", IN_COLAB, REPO_ROOT)